In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


ensure_family_result_dir = None


In [ ]:
def resolve_current_code_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    code_dir = Path.cwd().resolve()
    if code_dir.name != "1_Code":
        raise RuntimeError(
            "cannot resolve current code directory without __file__; "
            "current working directory must be the Step5 1_Code directory"
        )
    return code_dir


In [ ]:
def initialize_runtime() -> Path:
    global ensure_family_result_dir

    current_code_dir = resolve_current_code_dir()
    if str(current_code_dir) not in sys.path:
        sys.path.insert(0, str(current_code_dir))

    from _step5_shared import ensure_family_result_dir as shared_ensure_family_result_dir

    ensure_family_result_dir = shared_ensure_family_result_dir
    return current_code_dir


In [ ]:
def expected_output_files() -> list[str]:
    return [
        "4_family_metrics_summary.csv",
        "4_family_ranking.csv",
        "4_best_model_summary.csv",
    ]


In [ ]:
def sort_summary(summary: pd.DataFrame) -> pd.DataFrame:
    return summary.sort_values(
        ["WAPE_micro_mean", "R2_log_mean", "WAPE_macro_mean", "model_family", "model_id"],
        ascending=[True, False, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)


In [ ]:
def build_best_model_summary(summary: pd.DataFrame) -> pd.DataFrame:
    ranked = sort_summary(summary)
    tabpfn_rows = ranked.loc[
        ranked["model_family"].astype(str).str.contains("tabpfn", case=False, na=False)
    ].reset_index(drop=True)
    if tabpfn_rows.empty:
        raise ValueError("family summary does not contain any tabpfn rows")

    overall_best = ranked.iloc[0].copy()
    tabpfn_best = tabpfn_rows.iloc[0].copy()
    overall_best["selection_scope"] = "best_overall"
    overall_best["winner_note"] = "primary_metric_sorted"
    tabpfn_best["selection_scope"] = "best_tabpfn_anchor"
    tabpfn_best["winner_note"] = "tabpfn_anchor"
    return pd.DataFrame([overall_best, tabpfn_best])


In [ ]:
def load_family_summaries(step5_root: Path) -> pd.DataFrame:
    step5_root = Path(step5_root)
    ml_summary = pd.read_csv(step5_root / "2_Results" / "1_ml_family_compare" / "1_ml_metrics_by_model_mean_std.csv")
    dl_summary = pd.read_csv(step5_root / "2_Results" / "2_dl_family_compare" / "2_dl_metrics_by_model_mean_std.csv")
    tabpfn_summary = pd.read_csv(step5_root / "2_Results" / "3_tabpfn_family_compare" / "3_tabpfn_metrics_by_model_mean_std.csv")
    return pd.concat([ml_summary, dl_summary, tabpfn_summary], ignore_index=True)


In [ ]:
def write_collect_outputs(
    result_dir: Path,
    family_summary: pd.DataFrame,
    family_ranking: pd.DataFrame,
    best_model_summary: pd.DataFrame,
) -> None:
    if family_summary.empty or family_ranking.empty or best_model_summary.empty:
        raise RuntimeError("collect family ranking produced empty outputs")

    family_summary.to_csv(result_dir / "4_family_metrics_summary.csv", index=False)
    family_ranking.to_csv(result_dir / "4_family_ranking.csv", index=False)
    best_model_summary.to_csv(result_dir / "4_best_model_summary.csv", index=False)


In [ ]:
def _assert_runtime_outputs_written(result_dir: Path) -> None:
    expected_paths = [result_dir / file_name for file_name in expected_output_files()]
    missing_paths = [str(path) for path in expected_paths if not path.is_file()]
    if missing_paths:
        raise RuntimeError(f"missing collect runtime output files: {missing_paths}")

    for csv_path in expected_paths:
        frame = pd.read_csv(csv_path)
        if frame.empty:
            raise RuntimeError(f"runtime output must not be empty: {csv_path}")


In [ ]:
def run_collect_family_ranking(step5_root: Path) -> None:
    if ensure_family_result_dir is None:
        initialize_runtime()

    result_dir = ensure_family_result_dir(Path(step5_root), "collect")
    family_summary = load_family_summaries(step5_root)
    family_ranking = sort_summary(family_summary)
    best_model_summary = build_best_model_summary(family_ranking)
    write_collect_outputs(result_dir, family_summary, family_ranking, best_model_summary)


In [ ]:
# --- Step 1: Resolve Step5 root and output directory ---
current_code_dir = initialize_runtime()
step5_root = current_code_dir.parent
collect_result_dir = ensure_family_result_dir(step5_root, "collect")
print({"step": "collect_runtime_ready", "step5_root": str(step5_root), "result_dir": str(collect_result_dir)})


In [ ]:
# --- Step 2: Read family summary inputs ---
collect_family_summary = load_family_summaries(step5_root)
print({"step": "collect_inputs_loaded", "row_count": len(collect_family_summary), "families": sorted(collect_family_summary["model_family"].astype(str).unique().tolist())})


In [ ]:
# --- Step 3: Build ranking and best-model summary ---
collect_family_ranking = sort_summary(collect_family_summary)
collect_best_model_summary = build_best_model_summary(collect_family_ranking)
print(collect_best_model_summary)


In [ ]:
# --- Step 4: Write and verify collect outputs ---
write_collect_outputs(collect_result_dir, collect_family_summary, collect_family_ranking, collect_best_model_summary)
_assert_runtime_outputs_written(collect_result_dir)
print({"step": "collect_outputs_written", "files": expected_output_files(), "ranking_rows": len(collect_family_ranking)})
